In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import (
    LogisticRegression,
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    mean_squared_error,
    r2_score
)

from sklearn.preprocessing import LabelEncoder

import joblib

In [2]:
df = pd.read_csv("amazon_cleaned.csv")

print(df.shape)

display(df.head())

(1465, 18)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link,review_clean,review_tokens
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,look durable charging fine toono complains cha...,"['look', 'durable', 'charging', 'fine', 'toono..."
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...,ordered cable connect phone android auto car c...,"['ordered', 'cable', 'connect', 'phone', 'andr..."
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...,quite durable sturdy good nice product working...,"['quite', 'durable', 'sturdy', 'good', 'nice',..."
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...,good product long wire charge good nice bought...,"['good', 'product', 'long', 'wire', 'charge', ..."
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...,bought instead original apple work fast apple ...,"['bought', 'instead', 'original', 'apple', 'wo..."


In [4]:
# Review length feature
df["review_length"] = df["review_content"].apply(
    lambda x: len(str(x).split())
)

# Remove commas and currency symbols
df["actual_price"] = (
    df["actual_price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₹", "", regex=False)
)

df["discounted_price"] = (
    df["discounted_price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₹", "", regex=False)
)

# Convert to numeric
df["actual_price"] = pd.to_numeric(
    df["actual_price"],
    errors="coerce"
)

df["discounted_price"] = pd.to_numeric(
    df["discounted_price"],
    errors="coerce"
)

# -------------------------------------
# Price drop percentage
# -------------------------------------

df["price_drop_percentage"] = (
    (
        df["actual_price"]
        - df["discounted_price"]
    )
    / df["actual_price"]
) * 100

# -------------------------------------
# Sentiment Labels
# -------------------------------------

# 1 = positive
# 0 = negative

df["sentiment_label"] = df["rating"].apply(
    lambda x: 1 if x >= 3 else 0
)

# -------------------------------------
# Preview Features
# -------------------------------------

display(
    df[
        [
            "review_length",
            "price_drop_percentage",
            "sentiment_label"
        ]
    ].head()
)

,review_length,price_drop_percentage,sentiment_label
0,60,63.694268,1
1,201,42.979943,1
2,13,89.520800,1
3,77,52.932761,1
4,415,61.403509,1


In [14]:
print(df["sentiment_label"].value_counts())

sentiment_label
1    1459
0       6
Name: count, dtype: int64


In [5]:
le_category = LabelEncoder()

le_user = LabelEncoder()

df["category_encoded"] = le_category.fit_transform(
    df["category"].astype(str)
)

df["user_encoded"] = le_user.fit_transform(
    df["user_name"].astype(str)
)

display(
    df[
        [
            "category",
            "category_encoded",
            "user_name",
            "user_encoded"
        ]
    ].head()
)

,category,category_encoded,user_name,user_encoded
0,Computers&Accessories|Accessories&Peripherals|...,10,"Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...",522
1,Computers&Accessories|Accessories&Peripherals|...,10,"ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...",207
2,Computers&Accessories|Accessories&Peripherals|...,10,"Kunal,Himanshu,viswanath,sai niharka,saqib mal...",482
3,Computers&Accessories|Accessories&Peripherals|...,10,"Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...",614
4,Computers&Accessories|Accessories&Peripherals|...,10,"rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...",1139


In [6]:
tfidf = TfidfVectorizer(
    max_features=3000
)

X = tfidf.fit_transform(
    df["review_clean"]
)

y = df["sentiment_label"]

print(X.shape)

(1465, 3000)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1172, 3000)
(293, 3000)


In [8]:
lr = LogisticRegression()

lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)

print(
    classification_report(
        y_test,
        lr_preds
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      1.00      1.00       292

    accuracy                           1.00       293
   macro avg       0.50      0.50      0.50       293
weighted avg       0.99      1.00      0.99       293



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
dt = DecisionTreeClassifier()

dt.fit(X_train, y_train)

dt_preds = dt.predict(X_test)

print(
    classification_report(
        y_test,
        dt_preds
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      1.00      1.00       292

    accuracy                           0.99       293
   macro avg       0.50      0.50      0.50       293
weighted avg       0.99      0.99      0.99       293



In [10]:
rf = RandomForestClassifier()

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print(
    classification_report(
        y_test,
        rf_preds
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      1.00      1.00       292

    accuracy                           1.00       293
   macro avg       0.50      0.50      0.50       293
weighted avg       0.99      1.00      0.99       293



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
joblib.dump(
    lr,
    "logistic_regression.pkl"
)

joblib.dump(
    dt,
    "decision_tree.pkl"
)

joblib.dump(
    rf,
    "random_forest.pkl"
)

print("Models saved successfully.")

Models saved successfully.


In [12]:
y_reg = df["rating"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

regression_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso()
}

for name, model in regression_models.items():

    model.fit(
        X_train_reg,
        y_train_reg
    )

    preds = model.predict(X_test_reg)

    mse = mean_squared_error(
        y_test_reg,
        preds
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test_reg,
        preds
    )

    print("\n", name)

    print("MSE:", mse)

    print("RMSE:", rmse)

    print("R2:", r2)


 Linear Regression
MSE: 0.09875721275056157
RMSE: 0.3142566033523585
R2: -0.20943098472812838

 Ridge
MSE: 0.06260685330877774
RMSE: 0.25021361535451614
R2: 0.23328467724972335

 Lasso
MSE: 0.08207527169798136
RMSE: 0.2864878212035921
R2: -0.005135462077465736


## Regression Model Insights

Among the regression models tested, Ridge Regression achieved the best performance with the lowest RMSE and highest R² score. This suggests that regularization helped improve generalization when predicting product ratings from review text features. Overall, predicting exact ratings from customer reviews remains challenging due to the subjective nature of user feedback and variability in review language.

## Classification Model Insights

The sentiment classification models achieved extremely high overall accuracy across the dataset. Logistic Regression and Random Forest produced the strongest overall performance, while Decision Tree also classified positive reviews effectively.

However, the dataset was highly imbalanced, with 1,459 positive reviews and only 6 negative reviews. Due to this imbalance, the models struggled to learn meaningful patterns for the minority negative class. As a result, classification metrics for negative sentiment were very low despite high overall accuracy.

The imbalance demonstrates an important real-world machine learning challenge in e-commerce sentiment analysis, where customer reviews are often heavily skewed toward positive feedback. Future improvements could include balancing techniques such as oversampling, undersampling, or synthetic data generation to improve minority class prediction performance.

Overall, the models successfully demonstrated how TF-IDF vectorization and NLP preprocessing can support automated customer sentiment classification.